# Interactive HVDC-LCC \(V_{dc}\times I_{dc}\) characteristic

This notebook reproduces the **physical idea** of the figure where:

- the **rectifier is at its minimum firing angle** \(\alpha=\alpha_{\min}\);
- the **inverter controls the DC current** \(I_{dc}\);
- the inverter also has a **minimum extinction-angle boundary** \(\gamma=\gamma_{\min}\).

The simplified teaching model used here is

\[
V_{dc,R}=k_R V_{d0,R}\cos(\alpha_{\min})-R_{c,R}I_{dc}
\]

and, only to visualize the inverter \(\gamma_{\min}\) boundary,

\[
V_{\gamma}=k_I V_{d0,I}\cos(\gamma_{\min})-R_{c,I}I_{dc}.
\]

Because \(1~\Omega\times1~\mathrm{kA}=1~\mathrm{kV}\), the slope parameters
\(R_{c,R}\) and \(R_{c,I}\) can be entered directly in \(\Omega\).

> **Important:** this is a didactic steady-state characteristic, not a detailed electromagnetic-transient converter model. A complete LCC model also includes overlap, transformer ratio/taps, valve commutation equations, AC-system strength, controls, limits, and dynamics.


## What to observe

Move the sliders and watch the operating point.

1. **Reduce `Rectifier AC voltage`**: the rectifier characteristic moves downward.
2. **Reduce \(\alpha_{\min}\)**: the rectifier can produce a higher DC voltage.
3. **Increase `Rectifier commutation slope`**: the rectifier characteristic becomes steeper.
4. **Change `Inverter current order`**: the vertical current-control characteristic moves left/right.
5. **Change \(\gamma_{\min}\)**: the inverter extinction-angle boundary moves.
6. Enable **`Show normal-control comparison`** to see the higher rectifier current order used to illustrate the current-margin concept.

The red dot is the operating point represented by the original figure:

\[
I_{dc}=I_{\mathrm{ord,I}}
\]

\[
V_{dc}=V_{dc,R}(I_{\mathrm{ord,I}})
\]


In [1]:
# Imports and Colab widget support
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


In [2]:
# Interactive controls

rectifier_ac = widgets.FloatSlider(
    value=1.00, min=0.60, max=1.10, step=0.01,
    description="Rectifier AC voltage:",
    continuous_update=True,
    readout_format=".2f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

vd0_r = widgets.FloatSlider(
    value=500.0, min=300.0, max=700.0, step=5.0,
    description="Rectifier Vd0 [kV]:",
    continuous_update=True,
    readout_format=".0f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

alpha_min = widgets.FloatSlider(
    value=15.0, min=5.0, max=35.0, step=0.5,
    description="alpha_min [deg]:",
    continuous_update=True,
    readout_format=".1f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

rc_r = widgets.FloatSlider(
    value=18.0, min=0.0, max=60.0, step=1.0,
    description="Rectifier slope [ohm]:",
    continuous_update=True,
    readout_format=".1f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

i_order_i = widgets.FloatSlider(
    value=2.20, min=0.5, max=4.5, step=0.05,
    description="Inverter current order [kA]:",
    continuous_update=True,
    readout_format=".2f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

inverter_ac = widgets.FloatSlider(
    value=1.00, min=0.60, max=1.10, step=0.01,
    description="Inverter AC voltage:",
    continuous_update=True,
    readout_format=".2f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

vd0_i = widgets.FloatSlider(
    value=550.0, min=300.0, max=750.0, step=5.0,
    description="Inverter Vd0 [kV]:",
    continuous_update=True,
    readout_format=".0f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

gamma_min = widgets.FloatSlider(
    value=18.0, min=10.0, max=35.0, step=0.5,
    description="gamma_min [deg]:",
    continuous_update=True,
    readout_format=".1f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

rc_i = widgets.FloatSlider(
    value=10.0, min=0.0, max=60.0, step=1.0,
    description="Inverter slope [ohm]:",
    continuous_update=True,
    readout_format=".1f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

current_margin = widgets.FloatSlider(
    value=0.20, min=0.0, max=0.60, step=0.05,
    description="Current margin [kA]:",
    continuous_update=True,
    readout_format=".2f",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="520px")
)

show_normal = widgets.Checkbox(
    value=False,
    description="Show normal-control comparison",
    indent=False
)

show_gamma = widgets.Checkbox(
    value=True,
    description="Show gamma_min boundary",
    indent=False
)

reset_button = widgets.Button(
    description="Reset",
    button_style="",
    tooltip="Restore the initial values"
)


In [3]:
# Plot and explanation

plot_output = widgets.Output()

def rectifier_voltage(i_dc, k_ac, vd0, alpha_deg, r_comm):
    # Simplified rectifier minimum-alpha characteristic.
    return k_ac * vd0 * np.cos(np.deg2rad(alpha_deg)) - r_comm * i_dc

def inverter_gamma_boundary(i_dc, k_ac, vd0, gamma_deg, r_comm):
    # Simplified teaching approximation for the minimum-gamma boundary.
    return k_ac * vd0 * np.cos(np.deg2rad(gamma_deg)) - r_comm * i_dc

def redraw(*_):
    with plot_output:
        plot_output.clear_output(wait=True)

        i_max = 5.0
        i = np.linspace(0.0, i_max, 500)

        vr = rectifier_voltage(
            i, rectifier_ac.value, vd0_r.value, alpha_min.value, rc_r.value
        )
        vg = inverter_gamma_boundary(
            i, inverter_ac.value, vd0_i.value, gamma_min.value, rc_i.value
        )

        i_op = i_order_i.value
        v_op = rectifier_voltage(
            i_op, rectifier_ac.value, vd0_r.value, alpha_min.value, rc_r.value
        )
        v_gamma_at_op = inverter_gamma_boundary(
            i_op, inverter_ac.value, vd0_i.value, gamma_min.value, rc_i.value
        )

        fig, ax = plt.subplots(figsize=(12, 6.5))

        ax.plot(
            i, vr, linewidth=2.6,
            label=f"Rectifier: alpha = alpha_min = {alpha_min.value:.1f} deg"
        )

        if show_gamma.value:
            ax.plot(
                i, vg, linewidth=2.6,
                label=f"Inverter boundary: gamma = gamma_min = {gamma_min.value:.1f} deg"
            )

        # The vertical line is the current-controller characteristic.
        ax.axvline(
            i_op, linewidth=2.8,
            label=f"Inverter current control: Iord,I = {i_op:.2f} kA"
        )

        # Operating point is the intersection between inverter CC and rectifier alpha_min.
        ax.scatter([i_op], [v_op], s=95, zorder=5)
        ax.annotate(
            f"Operating point\nI = {i_op:.2f} kA\nV = {v_op:.1f} kV",
            xy=(i_op, v_op),
            xytext=(18, -58),
            textcoords="offset points",
            arrowprops={"arrowstyle": "->"},
            fontsize=10
        )

        if show_normal.value:
            i_order_r = i_order_i.value + current_margin.value
            ax.axvline(
                i_order_r,
                linestyle="--",
                linewidth=2.0,
                label=f"Rectifier CC reference: Iord,R = {i_order_r:.2f} kA"
            )

        ymax = max(
            100.0,
            np.nanmax(vr),
            np.nanmax(vg) if show_gamma.value else 0.0
        ) * 1.12

        ax.set_xlim(0.0, i_max)
        ax.set_ylim(0.0, ymax)
        ax.set_xlabel("DC current, Idc [kA]", fontsize=11)
        ax.set_ylabel("DC voltage, Vdc [kV]", fontsize=11)
        ax.set_title(
            "HVDC-LCC Vdc-Idc characteristic: inverter current control with rectifier at alpha_min",
            fontsize=12
        )
        ax.legend(loc="best")
        ax.grid(True, alpha=0.25)

        plt.show()

        separation = v_gamma_at_op - v_op
        if show_gamma.value:
            if separation >= 0:
                relation_text = (
                    "The operating point is below the plotted gamma_min boundary "
                    f"by {separation:.1f} kV in this simplified characteristic."
                )
            else:
                relation_text = (
                    "The operating point is above the plotted gamma_min boundary "
                    f"by {abs(separation):.1f} kV in this simplified characteristic. "
                    "This indicates that the selected control references are no longer "
                    "consistent with the intended graphical operating region."
                )
        else:
            relation_text = "The gamma_min boundary is hidden."

        display(HTML(
            f"""
            <div style="font-family:Arial; border:1px solid #bbb; padding:12px; border-radius:8px;">
              <b>Physical reading of the selected point</b><br><br>
              Inverter current controller fixes
              <b>Idc = {i_op:.2f} kA</b>.<br>
              Rectifier at <b>alpha_min = {alpha_min.value:.1f} deg</b> gives
              <b>Vdc = {v_op:.1f} kV</b> at that current.<br>
              Gamma-boundary voltage at the same current:
              <b>{v_gamma_at_op:.1f} kV</b>.<br><br>
              {relation_text}<br><br>
              <b>Cause-effect:</b><br>
              • Lower rectifier AC voltage shifts the entire rectifier characteristic downward.<br>
              • Lower alpha_min raises the rectifier voltage capability because cos(alpha) increases.<br>
              • A larger commutation slope produces a larger voltage drop as Idc increases.
            </div>
            """
        ))

def reset_values(_):
    # Restore the default values used to mimic the original schematic.
    rectifier_ac.value = 1.00
    vd0_r.value = 500.0
    alpha_min.value = 15.0
    rc_r.value = 18.0
    i_order_i.value = 2.20
    inverter_ac.value = 1.00
    vd0_i.value = 550.0
    gamma_min.value = 18.0
    rc_i.value = 10.0
    current_margin.value = 0.20
    show_normal.value = False
    show_gamma.value = True

for control in [
    rectifier_ac, vd0_r, alpha_min, rc_r, i_order_i,
    inverter_ac, vd0_i, gamma_min, rc_i, current_margin,
    show_normal, show_gamma
]:
    control.observe(redraw, names="value")

reset_button.on_click(reset_values)

left = widgets.VBox([
    widgets.HTML("<b>Rectifier</b>"),
    rectifier_ac,
    vd0_r,
    alpha_min,
    rc_r,
    widgets.HTML("<b>Current control</b>"),
    i_order_i,
    current_margin
])

right = widgets.VBox([
    widgets.HTML("<b>Inverter / gamma boundary</b>"),
    inverter_ac,
    vd0_i,
    gamma_min,
    rc_i,
    widgets.HTML("<b>Display options</b>"),
    show_gamma,
    show_normal,
    reset_button
])

controls = widgets.HBox(
    [left, right],
    layout=widgets.Layout(
        display="flex",
        flex_flow="row wrap",
        justify_content="space-between"
    )
)

display(controls, plot_output)
redraw()


Output()

## Suggested experiments

### Experiment A — Why the inverter controls current in this figure
Keep the inverter current order fixed and reduce **Rectifier AC voltage** from \(1.00\) to \(0.80\).

You should see the rectifier \(\alpha_{\min}\) characteristic move downward while the current-control line stays vertical at the same \(I_{dc}\). Therefore, the operating voltage changes while the inverter keeps the current approximately fixed.

### Experiment B — What \(\alpha_{\min}\) does
With the AC voltage fixed, move \(\alpha_{\min}\) from \(25^\circ\) toward \(10^\circ\).

Since \(\cos\alpha\) increases as \(\alpha\) decreases in this range, the rectifier DC-voltage capability rises.

### Experiment C — Why the characteristic slopes downward
Increase **Rectifier commutation slope**.

The term

\[
R_{c,R}I_{dc}
\]

increases, so the DC voltage falls faster with current.

### Experiment D — Current-margin concept
Enable **Show normal-control comparison**.

The dashed vertical line is placed at

\[
I_{\mathrm{ord,R}}=I_{\mathrm{ord,I}}+\Delta I.
\]

This visually introduces the current-margin concept used to coordinate rectifier and inverter current controllers.


## Engineering note

The notebook intentionally separates two ideas:

- **control characteristic**: the nearly vertical constant-current line;
- **converter physical limit/characteristic**: the \(\alpha_{\min}\) and \(\gamma_{\min}\) curves.

That distinction is the main physical message of the original diagram.

For detailed HVDC studies, the next step would be to replace the linearized characteristics with the full LCC commutation equations, including overlap \(\mu\), AC commutating voltage, transformer ratio/tap, commutation reactance, and the relation among \(\alpha\), \(\mu\), and \(\gamma\).
